# 04a — Supervised learning without leakage — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/04.md) · [Course map](../PLAN.md) · [Project](../../projects/stage04/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Build baselines and pipelines
- Compare regression and classification models
- Separate training, validation, and final testing


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Learning from examples without leaking answers

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A feature is information available to the model at prediction time. A label/target is what you want it to predict. Classification chooses a category; regression predicts a numeric value. Training fits parameters using examples. A baseline provides a simple comparison, such as the most frequent class or mean target.

Use training data to fit, validation data to select settings, and test data to estimate the selected method's performance. Related examples must remain in the same split. Leakage occurs when information unavailable at real prediction time influences training or selection.

A pipeline keeps learned preprocessing and the estimator together. For text, TF-IDF represents documents using word-weight features; fit its vocabulary on training data. A linear model combines weighted features; trees make branching decisions; forests average many trees; boosting adds successive models to reduce remaining error.

### Worked example: follow the values

Suppose a ticket's label is refund and a feature says resolution='refund issued'. That feature is available only after support resolves the ticket; it leaks the answer. Remove it even if validation accuracy drops. For regression predictions `[2,5]` and targets `[1,3]`, absolute errors are `[1,2]`, so MAE is 1.5.

### Pause and explain

Why should you not choose a model based on its test score?

<details><summary>Check your reasoning after trying</summary>

The test set then becomes part of model selection, making the reported score optimistic for that selection process.

</details>

### Common mistakes to check

A tiny instructional dataset proves mechanics, not useful real-world accuracy. Save the fitted preprocessing with the model.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

A feature is an input available at prediction time; a label is the target. Regression predicts a number; classification predicts a category. Training fits parameters. Validation chooses modeling decisions. The test set estimates performance after those choices are frozen. Never fit preprocessing on the test set.

Start with a baseline: mean prediction for regression or majority class for classification. Linear regression fits additive relationships; logistic regression maps a linear score to class probabilities. Trees partition feature space; random forests average trees; gradient boosting adds models that correct residual errors. k-nearest neighbors uses nearby examples; SVMs maximize a separating margin. Their inductive biases differ, so compare on the same evaluation protocol.

A pipeline fits preprocessing inside each cross-validation fold. Group related users/templates/documents together when splitting, and split by time when predicting the future. Duplicates across splits can make a weak model look excellent. Hyperparameter tuning must not repeatedly consult the final test set.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
print(pipeline)


## Exercise 1: Regression baseline

Predict the training-label mean for every test item. Reject empty training labels.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def mean_baseline(train_y, n):
    if len(train_y) == 0: raise ValueError("Empty training labels")
    return [sum(train_y)/len(train_y)]*n


In [ ]:
assert mean_baseline([2, 4, 6], 2) == [4, 4]
print("Exercise 1: checks passed")


**Why this works:** Even a baseline can leak if its mean comes from the test set.


## Exercise 2: Text pipeline

Return an unfitted TF-IDF + LogisticRegression pipeline with max_iter=500, random_state=42.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def text_pipeline():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=500, random_state=42))


In [ ]:
model = text_pipeline().fit(["login password", "account login", "refund invoice", "billing invoice"], [0, 0, 1, 1])
assert model.predict(["password login", "invoice refund"]).tolist() == [0, 1]
print("Exercise 2: checks passed")


**Why this works:** Vocabulary and IDF weights are learned transformations and must see training data only.


## Exercise 3: Group separation

Given rows with group and split keys, return True only if no group appears in multiple splits.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def groups_are_separate(rows):
    seen = {}
    for row in rows:
        group, split = row["group"], row["split"]
        if group in seen and seen[group] != split: return False
        seen[group] = split
    return True


In [ ]:
assert groups_are_separate([{"group": "a", "split": "train"}, {"group": "b", "split": "test"}])
assert not groups_are_separate([{"group": "a", "split": "train"}, {"group": "a", "split": "test"}])
print("Exercise 3: checks passed")


**Why this works:** Grouping by a shared source tests generalization beyond near-duplicate examples.


## Independent transfer

Train the support-ticket project with DummyClassifier, logistic regression, random forest, and gradient boosting. Fit a separate synthetic demand regression task with linear, tree, forest, and boosting models. Record MAE, RMSE, R² and classification macro-F1.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [04a interface and acceptance cases](../assignments/04a.md).
2. Copy the [blank starter](../assignments/starters/04a.py) to `work/assignments/04a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 04a --solution work/assignments/04a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/04a.md#04a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 04a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. What is leakage?

   Use of information unavailable at prediction time, including test-derived preprocessing or future labels.

2. Why can a simple model win?

   Limited data, strong linear signal, lower variance, and fewer tuning opportunities may favor it.


## Reading and review

- [Primary reference 1](https://scikit-learn.org/stable/common_pitfalls.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
